In [1]:
!pip install datasets pandas nltk beautifulsoup4 scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 13.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 34.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 34.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 36.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 38.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32/32 [datasets]/32 [datasets]ce-hub]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install

In [2]:
from datasets import load_dataset
import pandas as pd
import re
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


In [3]:
print("Loading dataset...")
ds = load_dataset("jason23322/high-accuracy-email-classifier")
df = ds['train'].to_pandas()

print(f"Dataset loaded: {len(df)} emails")
print(f"Columns: {list(df.columns)}")
df.head()

Loading dataset...


DatasetNotFoundError: Dataset 'jason23322/high-accuracy-email-classifier' is a gated dataset on the Hub. You must be authenticated to access it.

In [31]:
print("Dataset Info:")
print(df.info())

print("\nCategory Distribution:")
print(df['category'].value_counts())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10780 entries, 0 to 10779
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           10780 non-null  object
 1   subject      10780 non-null  object
 2   body         10780 non-null  object
 3   text         10780 non-null  object
 4   category     10780 non-null  object
 5   category_id  10780 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 505.4+ KB
None

Category Distribution:
category
forum           1800
verify_code     1800
promotions      1796
social_media    1796
spam            1794
updates         1794
Name: count, dtype: int64


In [32]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
print(f"Loaded {len(stop_words)} stopwords")

Loaded 198 stopwords


In [33]:
def preprocess_email(text):

    if pd.isna(text) or len(str(text).strip()) == 0:
        return ""
    
    text = BeautifulSoup(text, "html.parser").get_text()
    
    signature_patterns = [r'\nRegards,', r'\nThanks,', r'\nBest,', 
                         r'\nSincerely,', r'\nBest regards,']
    for pattern in signature_patterns:
        text = re.split(pattern, text, flags=re.IGNORECASE)[0]
    
    text = text.lower()
    
    text = re.sub(r'http\S+|www\S+', '', text)
    
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    return ' '.join(tokens)

print("Preprocessing function defined")

Preprocessing function defined


In [34]:
df['text'] = df['text'].apply(preprocess_email)
print("Preprocessing complete")

Preprocessing complete


In [35]:
df = df.drop_duplicates(subset=['text'], keep='first')
print(f"Dataset size after duplicate removal: {len(df)} emails")

Dataset size after duplicate removal: 8101 emails


In [36]:
def assign_urgency(row):
    text = str(row['text']).lower()
    subject = str(row['subject']).lower()
    combined = text + ' ' + subject
    
    high_keywords = ['urgent', 'immediately', 'asap', 'emergency', 'expires today', 
                     'unauthorized', 'fraud', 'suspended', 'blocked', 'critical', 
                     'action required', 'password', 'reset', 'alert', 'warning']
    
    medium_keywords = ['response needed', 'attention', 'due', 'schedule', 'appointment',
                      'meeting', 'invoice', 'payment', 'request', 'confirm']
    
    low_keywords = ['newsletter', 'news', 'weekly', 'announcement', 'tips', 
                   'offer', 'deal', 'promotion', 'welcome', 'notification']
    
    high_count = sum(1 for kw in high_keywords if kw in combined)
    medium_count = sum(1 for kw in medium_keywords if kw in combined)
    low_count = sum(1 for kw in low_keywords if kw in combined)
    
    if high_count > medium_count and high_count > low_count:
        return 'high'
    elif medium_count > low_count:
        return 'medium'
    else:
        return 'low'

df['urgency'] = df.apply(assign_urgency, axis=1)

print("Urgency Distribution:")
print(df['urgency'].value_counts().sort_index())
print(f"\nPercentages:")
for urgency in ['high', 'medium', 'low']:
    count = (df['urgency'] == urgency).sum()
    percentage = (count / len(df)) * 100
    print(f"{urgency.capitalize()}: {count} ({percentage:.1f}%)")

Urgency Distribution:
urgency
high       863
low       5384
medium    1854
Name: count, dtype: int64

Percentages:
High: 863 (10.7%)
Medium: 1854 (22.9%)
Low: 5384 (66.5%)


In [37]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['category'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['category'])

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

Train: 5670 (70.0%)
Validation: 1215 (15.0%)
Test: 1216 (15.0%)


In [38]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.8)

X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_val_tfidf = tfidf.transform(val_df['text'])
X_test_tfidf = tfidf.transform(test_df['text'])

import os
output_dir = 'module1_preprocessed_data'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(tfidf, f'{output_dir}/tfidf_vectorizer.pkl')
print("TF-IDF features created and saved")

TF-IDF features created and saved


In [39]:
import os
output_dir = 'module1_preprocessed_data'
os.makedirs(output_dir, exist_ok=True)

columns_to_save = ['text', 'category', 'category_id', 'urgency']

train_df[columns_to_save].to_csv(f'{output_dir}/train.csv', index=False)
val_df[columns_to_save].to_csv(f'{output_dir}/validation.csv', index=False)
test_df[columns_to_save].to_csv(f'{output_dir}/test.csv', index=False)

np.save(f'{output_dir}/X_train_tfidf.npy', X_train_tfidf.toarray())
np.save(f'{output_dir}/X_val_tfidf.npy', X_val_tfidf.toarray())
np.save(f'{output_dir}/X_test_tfidf.npy', X_test_tfidf.toarray())

np.save(f'{output_dir}/y_train_category.npy', train_df['category_id'].values)
np.save(f'{output_dir}/y_val_category.npy', val_df['category_id'].values)
np.save(f'{output_dir}/y_test_category.npy', test_df['category_id'].values)

print(f"All datasets saved to '{output_dir}/'")

All datasets saved to 'module1_preprocessed_data/'
